In [3]:
import numpy as np

from collections import deque

import matplotlib.pyplot as plt

%matplotlib inline

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
# Gym
import gymnasium as gym

# Hugging Face Hub
from huggingface_hub import (
    notebook_login,
)  # To log to our Hugging Face account to be able to upload models to the Hub.
import imageio


In [4]:
from ple import PLE
from ple.games.pixelcopter import Pixelcopter

# Create the environment
def create_pixelcopter_env():
    game = Pixelcopter()
    env = PLE(game, fps=30)  # Set display=False for headless
    return env

couldn't import doomish
Couldn't import doom


/Users/junsong/.pyenv/versions/ml_bank/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [5]:
# Preprocessing function (adjust based on your model's input requirements)
def preprocess_state(state):
    """
    Preprocess the game state for the neural network
    This should match the preprocessing used during training
    """
    if isinstance(state, np.ndarray) and len(state.shape) == 3:
        # If using image input
        state = np.transpose(state, (2, 0, 1))  # Channel first
        state = state / 255.0  # Normalize pixels
        return state.flatten()  # or keep as image depending on model
    else:
        # If using game state features
        return np.array(list(state.values()))


In [30]:
env = create_pixelcopter_env()
eval_env = create_pixelcopter_env()
env.init()
s = env.getGameState()
s

{'player_y': 24.0,
 'player_vel': 0,
 'player_dist_to_ceil': 7.0,
 'player_dist_to_floor': 17.0,
 'next_gate_dist_to_player': 48,
 'next_gate_block_top': 22,
 'next_gate_block_bottom': 31}

In [31]:
acts = env.getActionSet()
acts

[119, None]

In [ ]:
env.init()
for _ in range(10):
    s = env.getGameState()
    print(str(s) + "\n")
    action = 1
    reward = env.act(action)
    print(f"Reward: {reward} \n")

TypeError: PLE.init() takes 1 positional argument but 2 were given

In [7]:
s = preprocess_state(s)
s

array([24.,  0.,  7., 17., 48., 22., 31.])

In [8]:
s_size = s.shape[0]
a_size = 2


In [9]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [10]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, h_size * 2)
        self.fc3 = nn.Linear(h_size * 2, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)


In [19]:
pixelcopter_hyperparameters = {
    "h_size": 64,
    "n_training_episodes": 50000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-3,
    "env_id": "pixelcopter",
    "state_space": s_size,
    "action_space": a_size,
}


In [20]:
# Create policy and place it to the device
# torch.manual_seed(50)
pixelcopter_policy = Policy(
    pixelcopter_hyperparameters["state_space"],
    pixelcopter_hyperparameters["action_space"],
    pixelcopter_hyperparameters["h_size"],
).to(device)
pixelcopter_optimizer = optim.Adam(pixelcopter_policy.parameters(), lr=pixelcopter_hyperparameters["lr"])

In [18]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Help us to calculate the score during the training
    scores_deque = deque(maxlen=100)
    scores = []
    # Line 3 of pseudocode
    for i_episode in range(1, n_training_episodes + 1):
        saved_log_probs = []
        rewards = []
        env.init()

        # Line 4 of pseudocode
        for t in range(max_t):
            if env.game_over():
                break
            state = env.getGameState()
            state = preprocess_state(state)  # Apply your preprocessing
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            reward = env.act(action)
            rewards.append(reward)
            
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # Line 6 of pseudocode: calculate the return
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        # Compute the discounted returns at each timestep,
        for t in range(n_steps)[::-1]:
            disc_return_t = returns[0] if len(returns) > 0 else 0
            returns.appendleft(gamma * disc_return_t + rewards[t])

        ## standardization of the returns is employed to make training more stable
        eps = np.finfo(np.float32).eps.item()
        ## eps is the smallest representable float, which is
        # added to the standard deviation of the returns to avoid numerical instabilities
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # Line 7:
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # Line 8: PyTorch prefers gradient descent
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print(
                "Episode {}\tAverage Score: {:.2f}".format(
                    i_episode, np.mean(scores_deque)
                )
            )

    return scores


In [21]:
scores = reinforce(
    pixelcopter_policy,
    pixelcopter_optimizer,
    pixelcopter_hyperparameters["n_training_episodes"],
    pixelcopter_hyperparameters["max_t"],
    pixelcopter_hyperparameters["gamma"],
    1000,
)


Episode 1000	Average Score: 0.00
Episode 2000	Average Score: 0.00
Episode 3000	Average Score: 0.01
Episode 4000	Average Score: -0.01
Episode 5000	Average Score: 0.00
Episode 6000	Average Score: 0.00
Episode 7000	Average Score: 0.00
Episode 8000	Average Score: -0.01


KeyboardInterrupt: 